# 02 – Modellierung und Vergleich

**Eigenständige Bearbeitung – Aicha**

Untersucht werden XGBoost, LightGBM, CatBoost, SMOTE sowie XGBoost-Varianten mit Hyperparametervergleich und Top-20-Feature-Reduktion.

In [ ]:
import sys
sys.path.append('../../src')
from data_utils import load_arff, prepare_features, split_data, RANDOM_STATE
from evaluation import evaluate_model
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

df = load_arff('../../data/dataset.arff')
X, y, info = prepare_features(df)
X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print('Features nach Reduktion:', X.shape[1])
print('scale_pos_weight:', scale_pos_weight)

## 1. XGBoost-Baseline

Die Baseline verwendet `scale_pos_weight`, weil Klasse 1 nur ungefähr 3,6 % der Daten ausmacht.

In [ ]:
xgb = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                    subsample=0.8, colsample_bytree=0.8,
                    scale_pos_weight=scale_pos_weight, eval_metric='auc',
                    random_state=RANDOM_STATE, n_jobs=-1, tree_method='hist')
xgb.fit(X_train, y_train)
result_xgb = evaluate_model('XGBoost Baseline', xgb, X_val, y_val, X_test, y_test)
result_xgb

## 2. LightGBM

In [ ]:
X_lgb = X.copy()
cat_cols = info['cat']
for c in cat_cols:
    X_lgb[c] = X_lgb[c].fillna(-1).astype('int8')
lgb = LGBMClassifier(n_estimators=500, learning_rate=0.03, num_leaves=31,
                     subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                     scale_pos_weight=scale_pos_weight, random_state=RANDOM_STATE,
                     n_jobs=-1, verbosity=-1)
lgb.fit(X_lgb.loc[X_train.index], y_train, categorical_feature=cat_cols)
result_lgb = evaluate_model('LightGBM', lgb, X_lgb.loc[X_val.index], y_val, X_lgb.loc[X_test.index], y_test)
result_lgb

## 3. CatBoost

CatBoost wurde aus Laufzeitgründen auf einem stratifizierten Trainings-Subset von bis zu 200.000 Beobachtungen trainiert. Das wird bei der Interpretation berücksichtigt.

In [ ]:
X_cat = X.copy()
for c in cat_cols:
    X_cat[c] = X_cat[c].fillna(-1).astype(str)
cat_indices = [X_cat.columns.get_loc(c) for c in cat_cols]
sample_idx, _ = train_test_split(X_train.index, train_size=min(200000, len(X_train)),
                                  stratify=y_train, random_state=RANDOM_STATE)
cat = CatBoostClassifier(iterations=250, depth=6, learning_rate=0.05,
                         loss_function='Logloss', eval_metric='AUC', random_seed=RANDOM_STATE,
                         verbose=False, thread_count=-1, class_weights=[1.0, scale_pos_weight],
                         allow_writing_files=False)
cat.fit(X_cat.loc[sample_idx], y.loc[sample_idx], cat_features=cat_indices)
result_cat = evaluate_model('CatBoost (200k subset)', cat, X_cat.loc[X_val.index], y_val, X_cat.loc[X_test.index], y_test)
result_cat

## 4. XGBoost + SMOTE

SMOTE wird auf einem stratifizierten Trainings-Subset von bis zu 100.000 Beobachtungen untersucht. Ziel ist zu prüfen, ob Oversampling gegenüber Class Weighting Vorteile bringt.

In [ ]:
sample_idx, _ = train_test_split(X_train.index, train_size=min(100000, len(X_train)),
                                  stratify=y_train, random_state=RANDOM_STATE)
imp = SimpleImputer(strategy='median')
X_sm = imp.fit_transform(X.loc[sample_idx])
y_sm = y.loc[sample_idx]
X_res, y_res = SMOTE(random_state=RANDOM_STATE).fit_resample(X_sm, y_sm)
sm_xgb = XGBClassifier(n_estimators=250, max_depth=4, learning_rate=0.05,
                       subsample=0.8, colsample_bytree=0.8, eval_metric='auc',
                       random_state=RANDOM_STATE, n_jobs=-1, tree_method='hist')
sm_xgb.fit(X_res, y_res)
result_smote = evaluate_model('XGBoost + SMOTE (100k subset)', sm_xgb,
                              imp.transform(X_val), y_val, imp.transform(X_test), y_test)
result_smote

## 5. XGBoost-Hyperparametervergleich

In [ ]:
configs = [
    dict(n_estimators=500, max_depth=3, learning_rate=0.03, subsample=0.9, colsample_bytree=0.9, min_child_weight=5),
    dict(n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, min_child_weight=3),
    dict(n_estimators=400, max_depth=4, learning_rate=0.03, subsample=0.9, colsample_bytree=0.8, min_child_weight=5),
]
candidates=[]
for cfg in configs:
    m=XGBClassifier(**cfg, scale_pos_weight=scale_pos_weight, eval_metric='auc',
                    random_state=RANDOM_STATE, n_jobs=-1, tree_method='hist')
    m.fit(X_train,y_train)
    p=m.predict_proba(X_val)[:,1]
    candidates.append((__import__('sklearn').metrics.roc_auc_score(y_val,p),m))
best_model=max(candidates,key=lambda z:z[0])[1]
result_tuned=evaluate_model('XGBoost tuned',best_model,X_val,y_val,X_test,y_test)
result_tuned

## 6. Top-20-Features

Die Feature Importances des getunten XGBoost-Modells werden genutzt, um eine kompaktere Variante zu testen.

In [ ]:
importance = pd.DataFrame({'feature': X.columns, 'importance': best_model.feature_importances_}).sort_values('importance', ascending=False)
importance.head(20)

In [ ]:
top20 = importance.head(20)['feature'].tolist()
reduced = XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.03,
                        subsample=0.9, colsample_bytree=0.8, min_child_weight=5,
                        scale_pos_weight=scale_pos_weight, eval_metric='auc',
                        random_state=RANDOM_STATE, n_jobs=-1, tree_method='hist')
reduced.fit(X_train[top20], y_train)
result_top20=evaluate_model('XGBoost top-20 features', reduced, X_val[top20], y_val, X_test[top20], y_test)
result_top20

## 7. Zusammenfassung der Ergebnisse

Die bereits durchgeführten Läufe ergeben:

- XGBoost tuned: ROC-AUC **0,6394**
- XGBoost Baseline: ROC-AUC **0,6385**
- XGBoost Top-20: ROC-AUC **0,6383**, F1 **0,1162**
- CatBoost: ROC-AUC **0,6361**
- LightGBM: ROC-AUC **0,6202**
- XGBoost + SMOTE: ROC-AUC **0,6078**, Recall **0,3236**

Die Zahlen sind in `results/model_comparison.csv` und `results/metrics.json` gespeichert.

In [ ]:
results = pd.read_csv('../../results/model_comparison.csv')
display(results)